# Midterm Exam - Model A - Matthew Shaver

## Setup

In [ ]:
from pathlib import Path
import json
import random
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18

print('PyTorch version:', torch.__version__)

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'notebooks').exists():
    raise FileNotFoundError('Open this notebook from the project repository.')

print('Project root:', PROJECT_ROOT)

In [ ]:
SEED = 42
IMAGE_SIZE = 128
BATCH_SIZE = 32
NUM_WORKERS = 0
CLASS_NAMES = ('cats', 'dogs')

PRETRAINED = True
DROPOUT = 0.2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
FROZEN_EPOCHS = 10
TOTAL_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 8

SPLIT_DIR = PROJECT_ROOT / 'data' / 'splits'
CHECKPOINT_DIR = PROJECT_ROOT / 'models' / 'checkpoints'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'model_a'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print('Image size:', IMAGE_SIZE)
print('Batch size:', BATCH_SIZE)
print('Classes:', CLASS_NAMES)
print('Checkpoint directory:', CHECKPOINT_DIR)
print('Device:', DEVICE)

In [ ]:
def quarantine_checkpoint(checkpoint_dir):
    quarantine_root = CHECKPOINT_DIR / 'invalid_split'
    quarantine_root.mkdir(parents=True, exist_ok=True)
    destination = quarantine_root / checkpoint_dir.name
    copy_number = 1
    while destination.exists():
        destination = quarantine_root / f'{checkpoint_dir.name}_{copy_number}'
        copy_number += 1
    shutil.move(str(checkpoint_dir), str(destination))
    print('Moved incompatible checkpoint to:', destination)


phase_names = ['model_a_frozen', 'model_a']
for phase_name in phase_names:
    phase_dir = CHECKPOINT_DIR / phase_name
    print(
        phase_name,
        '| last:', (phase_dir / 'last.pt').exists(),
        '| best:', (phase_dir / 'best.pt').exists(),
        '| history:', (phase_dir / 'history.json').exists(),
    )

## Dataset

In [ ]:
SPLIT_DIR = PROJECT_ROOT / 'data' / 'splits'

train_table = pd.read_csv(SPLIT_DIR / 'train_split.csv')
val_table = pd.read_csv(SPLIT_DIR / 'val_split.csv')
test_table = pd.read_csv(SPLIT_DIR / 'test_split.csv')
split_tables = {
    'training': train_table,
    'validation': val_table,
    'test': test_table,
}

expected_classes = set(CLASS_NAMES)
SPLIT_METADATA = {}
for split_name, split_table in split_tables.items():
    class_counts = split_table['label'].value_counts().sort_index()
    observed_classes = set(class_counts.index)

    if observed_classes != expected_classes:
        raise ValueError(
            f'{split_name.title()} split must contain {sorted(expected_classes)}; '
            f'found {sorted(observed_classes)}. Rerun the data exploration notebook.'
        )

    split_counts = {}
    for class_name in CLASS_NAMES:
        split_counts[class_name] = int(class_counts.get(class_name, 0))
    SPLIT_METADATA[split_name] = split_counts

    print(f'{split_name.title()} images:', len(split_table))
    print(class_counts)


## Model A Functions

These functions keep the Model A data, model, and training logic in this notebook.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def load_rgb_image(image_path):
    with Image.open(image_path) as image:
        return image.convert('RGB')


def is_valid_image(image_path):
    try:
        with Image.open(image_path) as image:
            image.verify()
        return True
    except (FileNotFoundError, UnidentifiedImageError, OSError):
        return False


class ImageClassificationDataset(Dataset):
    def __init__(self, split_csv, class_names, transform, validate_images=False):
        self.records = pd.read_csv(split_csv)
        self.class_to_index = {
            class_name: index
            for index, class_name in enumerate(class_names)
        }
        self.transform = transform
        self.invalid_image_paths = []

        if validate_images:
            valid_rows = []
            for row_index, record in self.records.iterrows():
                image_path = Path(record['image_path'])
                if is_valid_image(image_path):
                    valid_rows.append(row_index)
                else:
                    self.invalid_image_paths.append(str(image_path))
            self.records = self.records.loc[valid_rows].reset_index(drop=True)

        if self.records.empty:
            raise ValueError(f'No valid images found in {split_csv}')

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records.iloc[index]
        image_path = Path(record['image_path'])
        image = load_rgb_image(image_path)
        image = self.transform(image)
        label_index = self.class_to_index[record['label']]
        label = torch.tensor(float(label_index), dtype=torch.float32)
        return image, label, str(image_path)


def get_classifier_transforms(image_size):
    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.RandomResizedCrop(image_size, scale=(0.85, 1.0)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    evaluation_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return {
        'train': train_transform,
        'validation': evaluation_transform,
        'test': evaluation_transform,
    }


def build_classification_loaders(split_dir, class_names, image_size, batch_size, num_workers):
    transform_map = get_classifier_transforms(image_size)
    train_dataset = ImageClassificationDataset(
        split_dir / 'train_split.csv',
        class_names,
        transform_map['train'],
    )
    val_dataset = ImageClassificationDataset(
        split_dir / 'val_split.csv',
        class_names,
        transform_map['validation'],
    )
    loader_options = {
        'batch_size': batch_size,
        'num_workers': num_workers,
        'pin_memory': DEVICE.type == 'cuda',
    }
    if num_workers > 0:
        loader_options['persistent_workers'] = True
    train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
    return train_loader, val_loader


class ResNet18Classifier(nn.Module):
    def __init__(self, pretrained, dropout):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = resnet18(weights=weights)
        feature_count = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_count, 1),
        )

    def forward(self, images):
        features = self.backbone(images)
        return self.classifier(features).squeeze(1)

    def freeze_backbone(self):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False

    def unfreeze_backbone(self):
        for parameter in self.parameters():
            parameter.requires_grad = True

    def trainable_parameters(self):
        return [
            parameter
            for parameter in self.parameters()
            if parameter.requires_grad
        ]


def calculate_metrics(logits, labels):
    predictions = (torch.sigmoid(logits) >= 0.5).to(torch.int64)
    targets = labels.to(torch.int64)
    true_positive = int(((predictions == 1) & (targets == 1)).sum())
    true_negative = int(((predictions == 0) & (targets == 0)).sum())
    false_positive = int(((predictions == 1) & (targets == 0)).sum())
    false_negative = int(((predictions == 0) & (targets == 1)).sum())
    total = max(int(targets.numel()), 1)
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {
        'accuracy': (true_positive + true_negative) / total,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


def run_epoch(model, loader, criterion, optimizer, phase_name, epoch_number, total_epochs):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    all_logits = []
    all_labels = []
    total_batches = len(loader)

    print(f'{phase_name} epoch {epoch_number}/{total_epochs} started ({total_batches} batches)')
    for batch_number, (images, labels, _) in enumerate(loader, start=1):
        images = images.to(DEVICE, non_blocking=DEVICE.type == 'cuda')
        labels = labels.to(DEVICE, non_blocking=DEVICE.type == 'cuda')

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = criterion(logits, labels)
            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        all_logits.append(logits.detach().cpu())
        all_labels.append(labels.detach().cpu())

        if batch_number == 1 or batch_number % 25 == 0 or batch_number == total_batches:
            print(f'{phase_name}: batch {batch_number}/{total_batches}')

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    metrics = calculate_metrics(logits, labels)
    metrics['loss'] = total_loss / len(loader.dataset)
    return metrics


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_loss, history):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_validation_loss': best_loss,
        'history': history,
    }
    torch.save(checkpoint, path)
    history_path = path.parent / 'history.json'
    history_path.write_text(json.dumps(history, indent=2), encoding='utf-8')


def train_classifier(model, train_loader, validation_loader, epochs, learning_rate, weight_decay, patience, checkpoint_dir, resume_from=None):
    model.to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.trainable_parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    history = []
    start_epoch = 0
    best_validation_loss = float('inf')
    epochs_without_improvement = 0

    if resume_from is not None and resume_from.exists():
        checkpoint = torch.load(resume_from, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        history = checkpoint['history']
        start_epoch = checkpoint['epoch'] + 1
        best_validation_loss = checkpoint['best_validation_loss']

    for epoch in range(start_epoch, epochs):
        epoch_start = time.perf_counter()
        epoch_number = epoch + 1
        train_metrics = run_epoch(model, train_loader, criterion, optimizer, 'Training', epoch_number, epochs)
        validation_metrics = run_epoch(model, validation_loader, criterion, None, 'Validation', epoch_number, epochs)
        scheduler.step(validation_metrics['loss'])

        epoch_metrics = {
            'epoch': epoch_number,
            'learning_rate': optimizer.param_groups[0]['lr'],
            'epoch_seconds': time.perf_counter() - epoch_start,
            **{f'train_{key}': value for key, value in train_metrics.items()},
            **{f'validation_{key}': value for key, value in validation_metrics.items()},
        }
        history.append(epoch_metrics)

        if validation_metrics['loss'] < best_validation_loss:
            best_validation_loss = validation_metrics['loss']
            epochs_without_improvement = 0
            save_checkpoint(checkpoint_dir / 'best.pt', model, optimizer, scheduler, epoch, best_validation_loss, history)
        else:
            epochs_without_improvement += 1

        save_checkpoint(checkpoint_dir / 'last.pt', model, optimizer, scheduler, epoch, best_validation_loss, history)
        print(f"Epoch {epoch_number}/{epochs} | train loss {train_metrics['loss']:.4f} | validation loss {validation_metrics['loss']:.4f} | validation accuracy {validation_metrics['accuracy']:.4f} | time {epoch_metrics['epoch_seconds']:.1f}s")

        if epochs_without_improvement >= patience:
            print('Early stopping triggered.')
            break

    best_checkpoint = checkpoint_dir / 'best.pt'
    if best_checkpoint.exists():
        checkpoint = torch.load(best_checkpoint, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
    return history


train_loader, val_loader = build_classification_loaders(
    SPLIT_DIR,
    CLASS_NAMES,
    IMAGE_SIZE,
    BATCH_SIZE,
    NUM_WORKERS,
)
batch_images, batch_labels, batch_paths = next(iter(train_loader))
print('Training batches:', len(train_loader))
print('Validation batches:', len(val_loader))
print('Batch image shape:', tuple(batch_images.shape))
print('Batch label shape:', tuple(batch_labels.shape))

In [ ]:
transform_map = get_classifier_transforms(image_size=IMAGE_SIZE)
sample_path = batch_paths[0]
sample_image = Image.open(sample_path).convert('RGB')
original_image = sample_image.resize((IMAGE_SIZE, IMAGE_SIZE))
augmented_tensors = [transform_map['train'](sample_image) for _ in range(3)]

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
augmented_images = [
    torch.clamp(image * std + mean, 0, 1).permute(1, 2, 0)
    for image in augmented_tensors
]

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(original_image)
axes[0].set_title('Original')
axes[0].axis('off')

for ax, image, title in zip(axes[1:], augmented_images, ['Aug 1', 'Aug 2', 'Aug 3']):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()

## ResNet18 baseline

Use a frozen pretrained backbone first.

In [ ]:
model = ResNet18Classifier(
    pretrained=PRETRAINED,
    dropout=DROPOUT,
)
model.freeze_backbone()

TOTAL_PARAMETERS = sum(parameter.numel() for parameter in model.parameters())
TRAINABLE_PARAMETERS = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(model)
print('Total parameters:', TOTAL_PARAMETERS)
print('Trainable parameters:', TRAINABLE_PARAMETERS)

In [ ]:
FROZEN_EPOCHS = 10
FROZEN_CHECKPOINT_DIR = CHECKPOINT_DIR / 'model_a_frozen'
frozen_metadata_path = FROZEN_CHECKPOINT_DIR / 'split_metadata.json'
frozen_resume = None

saved_split_metadata = None
if frozen_metadata_path.exists():
    saved_split_metadata = json.loads(frozen_metadata_path.read_text(encoding='utf-8'))

frozen_last_path = FROZEN_CHECKPOINT_DIR / 'last.pt'
frozen_checkpoint_files = list(FROZEN_CHECKPOINT_DIR.glob('*.pt'))
if frozen_checkpoint_files:
    if saved_split_metadata == SPLIT_METADATA and frozen_last_path.exists():
        frozen_resume = frozen_last_path
        print('Resuming frozen training from:', frozen_resume)
    elif saved_split_metadata != SPLIT_METADATA:
        quarantine_checkpoint(FROZEN_CHECKPOINT_DIR)

FROZEN_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
frozen_metadata_path.write_text(
    json.dumps(SPLIT_METADATA, indent=2),
    encoding='utf-8',
)

print(f'Starting frozen training for {FROZEN_EPOCHS} epochs.', flush=True)
history = train_classifier(
    model=model,
    train_loader=train_loader,
    validation_loader=val_loader,
    epochs=FROZEN_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    patience=EARLY_STOPPING_PATIENCE,
    checkpoint_dir=FROZEN_CHECKPOINT_DIR,
    resume_from=frozen_resume,
)

pd.DataFrame(history)

In [ ]:
def plot_learning(history, title, output_path=None):
    metrics = pd.DataFrame(history)
    epochs = metrics['epoch']

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, metrics['train_loss'], label='train loss')
    plt.plot(epochs, metrics['validation_loss'], label='val loss')
    plt.title(title + ' - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, metrics['train_accuracy'], label='train accuracy')
    plt.plot(epochs, metrics['validation_accuracy'], label='val accuracy')
    plt.title(title + ' - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    if output_path is not None:
        plt.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.show()


plot_learning(history, 'ResNet18 baseline')

## Fine-tune ResNet18

Unfreeze the backbone and train again with a smaller learning rate.

In [ ]:
model.unfreeze_backbone()

FINE_TUNE_EPOCHS = TOTAL_EPOCHS - FROZEN_EPOCHS
FINE_TUNE_CHECKPOINT_DIR = CHECKPOINT_DIR / 'model_a'
fine_tune_metadata_path = FINE_TUNE_CHECKPOINT_DIR / 'split_metadata.json'
fine_tune_resume = None

saved_split_metadata = None
if fine_tune_metadata_path.exists():
    saved_split_metadata = json.loads(fine_tune_metadata_path.read_text(encoding='utf-8'))

fine_tune_last_path = FINE_TUNE_CHECKPOINT_DIR / 'last.pt'
fine_tune_checkpoint_files = list(FINE_TUNE_CHECKPOINT_DIR.glob('*.pt'))
if fine_tune_checkpoint_files:
    if saved_split_metadata == SPLIT_METADATA and fine_tune_last_path.exists():
        fine_tune_resume = fine_tune_last_path
        print('Resuming fine-tuning from:', fine_tune_resume)
    elif saved_split_metadata != SPLIT_METADATA:
        quarantine_checkpoint(FINE_TUNE_CHECKPOINT_DIR)

FINE_TUNE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
fine_tune_metadata_path.write_text(
    json.dumps(SPLIT_METADATA, indent=2),
    encoding='utf-8',
)

TRAINABLE_PARAMETERS = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print('Trainable parameters:', TRAINABLE_PARAMETERS)
print('Learning rate:', LEARNING_RATE / 10)
print('Resume checkpoint:', fine_tune_resume)

In [ ]:
print(f'Starting fine-tuning for {FINE_TUNE_EPOCHS} epochs.', flush=True)
history = train_classifier(
    model=model,
    train_loader=train_loader,
    validation_loader=val_loader,
    epochs=FINE_TUNE_EPOCHS,
    learning_rate=LEARNING_RATE / 10,
    weight_decay=WEIGHT_DECAY,
    patience=EARLY_STOPPING_PATIENCE,
    checkpoint_dir=FINE_TUNE_CHECKPOINT_DIR,
    resume_from=fine_tune_resume,
)

pd.DataFrame(history).tail()

In [ ]:
plot_learning(
    history,
    'ResNet18 fine-tuned',
    OUTPUT_DIR / 'training_history.png',
)

print('Training history saved to:', OUTPUT_DIR / 'training_history.png')

## Test Model A

Load the best fine-tuned checkpoint and evaluate the untouched test data.

In [ ]:
BEST_MODEL_PATH = FINE_TUNE_CHECKPOINT_DIR / 'best.pt'
if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError('Complete Model A training before running the test evaluation.')
if not fine_tune_metadata_path.exists():
    raise RuntimeError('Checkpoint metadata is missing. Rerun Model A training with the valid splits.')
saved_split_metadata = json.loads(fine_tune_metadata_path.read_text(encoding='utf-8'))
if saved_split_metadata != SPLIT_METADATA:
    raise RuntimeError('Checkpoint data does not match the current splits. Rerun Model A training.')

checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE)
model.eval()

test_dataset = ImageClassificationDataset(
    split_csv=SPLIT_DIR / 'test_split.csv',
    class_names=CLASS_NAMES,
    transform=get_classifier_transforms(image_size=IMAGE_SIZE)['test'],
    validate_images=True,
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == 'cuda',
)

valid_test_classes = set(test_dataset.records['label'])
if valid_test_classes != expected_classes:
    raise ValueError(
        f'Valid test images must contain {sorted(expected_classes)}; '
        f'found {sorted(valid_test_classes)}. Rerun the data exploration notebook.'
    )

print('Loaded model:', BEST_MODEL_PATH)
print('Test images:', len(test_dataset))
print('Test batches:', len(test_loader))
print('Skipped invalid test images:', len(test_dataset.invalid_image_paths))
for invalid_path in test_dataset.invalid_image_paths:
    print('Skipped:', Path(invalid_path).name)

In [ ]:
all_probabilities = []
all_targets = []
all_paths = []

with torch.inference_mode():
    for images, labels, image_paths in test_loader:
        logits = model(images.to(DEVICE, non_blocking=DEVICE.type == 'cuda'))
        all_probabilities.append(torch.sigmoid(logits).cpu())
        all_targets.append(labels.cpu())
        all_paths.extend(image_paths)

probabilities = torch.cat(all_probabilities).numpy()
targets = torch.cat(all_targets).numpy().astype(int)
predictions = (probabilities >= 0.5).astype(int)

expected_target_indices = {0, 1}
observed_target_indices = set(targets.tolist())
if observed_target_indices != expected_target_indices:
    raise ValueError(
        f'Evaluation requires target indices {sorted(expected_target_indices)}; '
        f'found {sorted(observed_target_indices)}.'
    )

target_counts = pd.Series(targets).value_counts().reindex([0, 1], fill_value=0)
prediction_counts = pd.Series(predictions).value_counts().reindex([0, 1], fill_value=0)
print('Target counts:', {CLASS_NAMES[index]: int(target_counts[index]) for index in [0, 1]})
print('Prediction counts:', {CLASS_NAMES[index]: int(prediction_counts[index]) for index in [0, 1]})

false_positive_rate, true_positive_rate, _ = roc_curve(targets, probabilities)
test_metrics = {
    'accuracy': float(accuracy_score(targets, predictions)),
    'precision': float(precision_score(targets, predictions, zero_division=0)),
    'recall': float(recall_score(targets, predictions, zero_division=0)),
    'f1': float(f1_score(targets, predictions, zero_division=0)),
    'roc_auc': float(auc(false_positive_rate, true_positive_rate)),
}

print(pd.Series(test_metrics).round(4))

In [ ]:
prediction_table = pd.DataFrame({
    'image_path': all_paths,
    'actual': [CLASS_NAMES[index] for index in targets],
    'probability_dog': probabilities,
    'predicted': [CLASS_NAMES[index] for index in predictions],
})

(OUTPUT_DIR / 'test_metrics.json').write_text(
    json.dumps(test_metrics, indent=2),
    encoding='utf-8',
)
prediction_table.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay(
    confusion_matrix(targets, predictions, labels=[0, 1]),
    display_labels=CLASS_NAMES,
).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Model A - Confusion Matrix')

axes[1].plot(
    false_positive_rate,
    true_positive_rate,
    label=f"AUC = {test_metrics['roc_auc']:.3f}",
)
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[1].set_title('Model A - ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_evaluation.png', dpi=200, bbox_inches='tight')
plt.show()

print('Results saved to:', OUTPUT_DIR)

## Results

The test metrics, predictions, learning curve, confusion matrix, and ROC curve are saved in the Model A output folder.

## Checkpoints

`last.pt`, `best.pt`, and `history.json` are saved locally after every epoch. Rerunning the training cells resumes from `last.pt`.